# Tracking a protein backbone: orientation along the chain from torsion angles

**Task.** A protein backbone is a chain of rigid residues. The orientation of residue j relative to residue i is
the **ordered product** of the rotations between consecutive residues, and each of those is fixed by three torsion
angles (ψ, ω, φ) interleaved with nearly constant bond angles. Given only the torsion angles from residue i to
residue j, predict the relative orientation of the two residues' frames (the frames AlphaFold uses: origin at Cα,
axes from the N, Cα and C atoms). Ground truth comes from real crystal structures in the Protein Data Bank.
Models are trained on stretches of **1–16 residues** and tested on stretches of up to **256 residues** of
**held-out proteins**.

**Models** (same inputs: a sequence of tokens, each a token type and an angle):
* **HyperSSM**: one quaternion channel. Each token is either a torsion (it turns the state by `exp(α w_type)`, a
  rotation by its angle about a learned axis) or a bond (a learned constant rotation): `v = W (type ⊗ [1, α])`,
  48 parameters in all. The ordered product of the tokens is the predicted orientation. 4 numbers of state.
* **ℂ diagonal (selective)**: Mamba-style, 64 complex channels, MLP readout. Commutative; 128 numbers of state.
* **LSTM**: 128 units, linear output.
* **Ideal geometry** (no learning): rebuild the backbone from the torsions with textbook bond lengths and angles
  (Engh & Huber) and read off the orientation, the counterpart of integrating a gyroscope.
* **Ideal geometry, wrong order** (no learning): the same per-residue rotations multiplied in reverse order.
* **No rotation**: always predict the identity.

Loss `1 − ⟨q̂, q⟩²` on the final orientation (sign-invariant); error = rotation angle between prediction and truth.

**Multi-start.** A product of rotations has local minima: fitted to exact ideal-geometry targets, HyperSSM reaches
0.0° from some initializations and ~32° from others. Every model therefore tries `N_INIT` initializations briefly on
1–2 residues and keeps the one with the lowest training loss (no validation or test data), then trains fully.

In [ ]:
import os, math, time, json, glob, random, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# ---------------- configuration ----------------
PDB_DIR       = "pdb_files"
LOCAL_ONLY    = False            # True: use the files already in PDB_DIR (no download)
N_PROTEINS    = 600              # entries sampled from the PDB search
MAX_RES       = 1.8              # X-ray resolution cutoff (Å)
SPLIT         = (0.8, 0.1, 0.1)  # train / validation / test, by entry
TRAIN_RES     = (1, 16)          # residues per training stretch
VAL_RES       = 64
TEST_RES      = [1, 2, 4, 8, 16, 32, 64, 128, 256]
N_TEST, N_VAL = 512, 256
MODELS        = ["HyperSSM", "C-diagonal", "LSTM"]
SEEDS         = [0, 1, 2]
STEPS         = 3_000
BATCH         = 128
LR            = {"HyperSSM": 1e-2, "C-diagonal": 1e-3, "LSTM": 1e-3}
EVAL_EVERY    = 250
SEED_DATA     = 0
CURRICULUM    = 0.3              # fraction of training on stretches of 1–2 residues (same for every model)
N_INIT        = 8                # multi-start: random initializations tried for every model ...
WARMUP_STEPS  = 300              # ... each for this many steps on 1–2 residues; the lowest TRAINING loss is kept

## Data: high-resolution crystal structures from the PDB

In [ ]:
os.makedirs(PDB_DIR, exist_ok=True)
if not LOCAL_ONLY:
    query = {
        "query": {"type": "group", "logical_operator": "and", "nodes": [
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_entry_info.resolution_combined", "operator": "less_or_equal", "value": MAX_RES}},
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "exptl.method", "operator": "exact_match", "value": "X-RAY DIFFRACTION"}},
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_entry_info.polymer_entity_count_protein", "operator": "equals", "value": 1}},
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_entry_info.deposited_polymer_monomer_count", "operator": "range",
                "value": {"from": 150, "to": 800}}}]},
        "return_type": "entry",
        "request_options": {"paginate": {"start": 0, "rows": 10000}, "results_content_type": ["experimental"]}}
    req = urllib.request.Request("https://search.rcsb.org/rcsbsearch/v2/query", data=json.dumps(query).encode(),
                                 headers={"Content-Type": "application/json"})
    ids = [r["identifier"] for r in json.load(urllib.request.urlopen(req))["result_set"]]
    ids = sorted(np.random.default_rng(SEED_DATA).choice(ids, size=min(N_PROTEINS, len(ids)), replace=False))
    def fetch(pid):
        path = os.path.join(PDB_DIR, pid + ".pdb")
        if not os.path.exists(path):
            try: urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pid}.pdb", path)
            except Exception: return None
        return path
    with ThreadPoolExecutor(16) as ex: list(ex.map(fetch, ids))
FILES = sorted(glob.glob(os.path.join(PDB_DIR, "*.pdb")))
print(f"{len(FILES)} structure files")

def parse_backbone(path, min_len=20):
    """Contiguous backbone segments (N, CA, C coordinates) of the first model; alternate locations: ' ' or 'A'.
    A segment ends wherever the peptide bond C(i)–N(i+1) is missing or longer than 1.7 Å."""
    chains = {}
    for line in open(path, errors="ignore"):
        if line.startswith("ENDMDL"): break
        if not line.startswith("ATOM"): continue
        name, alt = line[12:16].strip(), line[16]
        if name not in ("N", "CA", "C") or alt not in (" ", "A"): continue
        key = (line[22:26], line[26])
        chains.setdefault(line[21], {}).setdefault(key, {})[name] = (float(line[30:38]), float(line[38:46]),
                                                                    float(line[46:54]))
    segs = []
    for res in chains.values():
        X = np.array([[r["N"], r["CA"], r["C"]] for r in res.values() if len(r) == 3])
        if len(X) < min_len: continue
        gap = np.linalg.norm(X[1:, 0] - X[:-1, 2], axis=1) > 1.7
        for s in np.split(X, np.where(gap)[0] + 1):
            if len(s) >= min_len: segs.append(s)
    return segs

def frames(X):
    """Residue frames (AlphaFold convention): columns e1 = Cα→C, e2 in the N–Cα–C plane, e3 = e1 × e2."""
    N, CA, C = X[:, 0], X[:, 1], X[:, 2]
    e1 = C - CA; e1 /= np.linalg.norm(e1, axis=-1, keepdims=True)
    u = N - CA; e2 = u - (u * e1).sum(-1, keepdims=True) * e1; e2 /= np.linalg.norm(e2, axis=-1, keepdims=True)
    return np.stack([e1, e2, np.cross(e1, e2)], -1)

def dihedral(a, b, c, d):
    b0, b1, b2 = a - b, c - b, d - c
    b1n = b1 / np.linalg.norm(b1, axis=-1, keepdims=True)
    v = b0 - (b0 * b1n).sum(-1, keepdims=True) * b1n; w = b2 - (b2 * b1n).sum(-1, keepdims=True) * b1n
    return np.arctan2((np.cross(b1n, v) * w).sum(-1), (v * w).sum(-1))

def torsions(X):
    """For each step k → k+1: (ψ_k, ω_k, φ_{k+1})."""
    N, CA, C = X[:, 0], X[:, 1], X[:, 2]
    psi = dihedral(N[:-1], CA[:-1], C[:-1], N[1:])
    omega = dihedral(CA[:-1], C[:-1], N[1:], CA[1:])
    phi = dihedral(C[:-1], N[1:], CA[1:], C[1:])
    return np.stack([psi, omega, phi], -1).astype(np.float32)

def rot_to_quat(R):
    """Rotation matrices (..., 3, 3) -> unit quaternions (..., 4), w first."""
    m = R; t = m[..., 0, 0] + m[..., 1, 1] + m[..., 2, 2]
    q = np.stack([1 + t, 1 + m[..., 0, 0] - m[..., 1, 1] - m[..., 2, 2],
                  1 - m[..., 0, 0] + m[..., 1, 1] - m[..., 2, 2], 1 - m[..., 0, 0] - m[..., 1, 1] + m[..., 2, 2]], -1)
    k = np.argmax(q, -1)
    out = np.zeros(m.shape[:-2] + (4,))
    for i in range(4):
        sel = k == i; mm = m[sel]; s = np.sqrt(np.maximum(q[sel, i], 1e-12)) * 2
        if i == 0: out[sel] = np.stack([s / 4, (mm[:, 2, 1] - mm[:, 1, 2]) / s, (mm[:, 0, 2] - mm[:, 2, 0]) / s, (mm[:, 1, 0] - mm[:, 0, 1]) / s], -1)
        if i == 1: out[sel] = np.stack([(mm[:, 2, 1] - mm[:, 1, 2]) / s, s / 4, (mm[:, 0, 1] + mm[:, 1, 0]) / s, (mm[:, 0, 2] + mm[:, 2, 0]) / s], -1)
        if i == 2: out[sel] = np.stack([(mm[:, 0, 2] - mm[:, 2, 0]) / s, (mm[:, 0, 1] + mm[:, 1, 0]) / s, s / 4, (mm[:, 1, 2] + mm[:, 2, 1]) / s], -1)
        if i == 3: out[sel] = np.stack([(mm[:, 1, 0] - mm[:, 0, 1]) / s, (mm[:, 0, 2] + mm[:, 2, 0]) / s, (mm[:, 1, 2] + mm[:, 2, 1]) / s, s / 4], -1)
    return out / np.linalg.norm(out, axis=-1, keepdims=True)

entries = {}
for f in FILES:
    segs = parse_backbone(f)
    if segs: entries[os.path.basename(f)[:-4]] = [dict(X=s, R=frames(s), tor=torsions(s)) for s in segs]
names = sorted(entries); random.Random(SEED_DATA).shuffle(names)
n_tr = int(SPLIT[0] * len(names)); n_va = max(1, int(SPLIT[1] * len(names)))
SPLITS = {"train": names[:n_tr], "val": names[n_tr:n_tr + n_va], "test": names[n_tr + n_va:]}
for k, v in SPLITS.items():
    segs = [s for n in v for s in entries[n]]
    print(f"{k}: {len(v)} entries, {len(segs)} segments, {sum(len(s['X']) for s in segs)} residues")

## Windows and tokens
A window from residue i to residue i+n becomes 6n+1 tokens: for each step, a bond token, ψ, a bond token, ω, a
bond token, φ, and a final bond token. Token types: 0 first bond, 1 ψ, 2 bond, 3 ω, 4 bond, 5 φ, 6 bond between
residues, 7 last bond. Each token carries its angle (0 for bond tokens).

In [ ]:
N_TYPES = 8
def window_tokens(tor):
    """tor: (n, 3) torsions for n steps -> types (6n+1,), angles (6n+1,)."""
    n = len(tor); types, ang = [], []
    for k in range(n):
        types += [0 if k == 0 else 6, 1, 2, 3, 4, 5]; ang += [0.0, tor[k, 0], 0.0, tor[k, 1], 0.0, tor[k, 2]]
    types.append(7); ang.append(0.0)
    return np.array(types), np.array(ang, np.float32)

def make_windows(split, n_res, count, rng):
    """count windows of n_res steps (an int, or a (lo, hi) range) from random segments of the split."""
    segs = [s for n in SPLITS[split] for s in entries[n]]
    lo, hi = (n_res, n_res) if np.isscalar(n_res) else n_res
    T_, A_, Q_, starts = [], [], [], []
    tries = 0
    while len(T_) < count and tries < 100 * count:
        tries += 1
        s = segs[rng.integers(len(segs))]; n = int(rng.integers(lo, hi + 1))
        if len(s["X"]) < n + 1: continue
        i = int(rng.integers(0, len(s["X"]) - n))
        R = s["R"][i].T @ s["R"][i + n]                                   # orientation of residue i+n in frame i
        t, a = window_tokens(s["tor"][i:i + n])
        T_.append(t); A_.append(a); Q_.append(rot_to_quat(R)); starts.append((s, i, n))
    if not T_:
        raise ValueError(f"no segment of the {split} split is longer than {hi} residues")
    L = np.array([len(t) for t in T_]); Tm = L.max()
    types = np.full((len(T_), Tm), -1); ang = np.zeros((len(T_), Tm), np.float32)
    for b, (t, a) in enumerate(zip(T_, A_)): types[b, :len(t)] = t; ang[b, :len(a)] = a
    return (torch.from_numpy(types), torch.from_numpy(ang), torch.from_numpy(L),
            torch.from_numpy(np.array(Q_, np.float32)), starts)

## Models

In [ ]:
def qmul(a, b):
    aw, ax, ay, az = a.unbind(-1); bw, bx, by, bz = b.unbind(-1)
    return torch.stack([aw*bw - ax*bx - ay*by - az*bz, aw*bx + ax*bw + ay*bz - az*by,
                        aw*by - ax*bz + ay*bw + az*bx, aw*bz + ax*by - ay*bx + az*bw], -1)
def qexp(v):
    th = v.norm(dim=-1, keepdim=True)
    return torch.cat([torch.cos(th), torch.sinc(th / math.pi) * v], -1)
def ordered_product(a):
    """a_1 ⊗ a_2 ⊗ ⋯ ⊗ a_T (frames compose along the chain), log-depth tree as in a parallel scan.
    (The HyperSSM recurrence multiplies on the left; reading out the conjugate gives this order.)"""
    B, T, _ = a.shape; T2 = 1 << max(0, (T - 1).bit_length())
    if T2 > T:
        e = a.new_zeros(B, T2 - T, 4); e[..., 0] = 1; a = torch.cat([a, e], 1)
    while a.shape[1] > 1: a = qmul(a[:, 0::2], a[:, 1::2])
    return a[:, 0]

def features(types, ang):
    """generic token features for the baselines: one-hot type, angle, cos, sin (zeros for padding)."""
    valid = (types >= 0).float().unsqueeze(-1)
    oh = F.one_hot(types.clamp(min=0), N_TYPES).float()
    return torch.cat([oh, ang[..., None], torch.cos(ang)[..., None], torch.sin(ang)[..., None]], -1) * valid

class HyperSSMBackbone(nn.Module):
    """v = W (type ⊗ [1, α]): bond tokens give a constant rotation, torsion tokens a rotation linear in the angle."""
    def __init__(self):
        super().__init__()
        W = torch.zeros(N_TYPES, 2, 3)
        u = F.normalize(torch.randn(N_TYPES, 3), dim=-1)
        W[[1, 3, 5], 1] = 0.5 * u[[1, 3, 5]]           # torsions start as rotations by their angle, random axes
        W[[0, 2, 4, 6, 7], 0] = 0.3 * torch.randn(5, 3)  # bonds start as random small rotations
        self.W = nn.Parameter(W)
    def forward(self, types, ang, L):
        valid = (types >= 0).float().unsqueeze(-1)
        Wt = self.W[types.clamp(min=0)]                                     # (B, T, 2, 3)
        v = (Wt[:, :, 0] + ang[..., None] * Wt[:, :, 1]) * valid
        return ordered_product(qexp(v))

class CDiagonal(nn.Module):
    """Selective complex diagonal SSM (commutative transitions): h_k = exp(Δ_k(−α + iθ_k)) h_{k−1} + Δ_k B x_k."""
    def __init__(self, K=64, hidden=128, d_in=N_TYPES + 3):
        super().__init__(); self.K = K
        self.dt = nn.Linear(d_in, K); self.theta = nn.Linear(d_in, K); self.B = nn.Linear(d_in, 2 * K)
        self.log_alpha = nn.Parameter(torch.log(torch.rand(K) * 0.1 + 1e-3))
        self.head = nn.Sequential(nn.Linear(2 * K, hidden), nn.GELU(), nn.Linear(hidden, 4))
    def forward(self, types, ang, L, chunk=64):
        outs = []
        for i in range(0, types.shape[0], chunk):
            x = features(types[i:i + chunk], ang[i:i + chunk]); m = (types[i:i + chunk] >= 0).float().unsqueeze(-1)
            d = F.softplus(self.dt(x)) * m
            log_a = torch.complex(-d * self.log_alpha.exp(), d * self.theta(x))
            Bx = torch.view_as_complex(self.B(x).view(*x.shape[:2], self.K, 2).contiguous()) * d
            S = torch.cumsum(log_a, 1)
            h = (torch.exp(S[:, -1:] - S) * Bx).sum(1)
            outs.append(self.head(torch.cat([h.real, h.imag], -1)))
        return F.normalize(torch.cat(outs), dim=-1)

class LSTMBackbone(nn.Module):
    def __init__(self, hidden=128):
        super().__init__(); self.lstm = nn.LSTM(N_TYPES + 3, hidden, batch_first=True); self.head = nn.Linear(hidden, 4)
    def forward(self, types, ang, L):
        out, _ = self.lstm(features(types, ang))
        return F.normalize(self.head(out[torch.arange(len(L)), L - 1]), dim=-1)

MAKE = {"HyperSSM": HyperSSMBackbone, "C-diagonal": CDiagonal, "LSTM": LSTMBackbone}

# ---- references without learning: rebuild the backbone with ideal geometry (Engh & Huber 1991) ----
BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329)
ANGLE = dict(N_CA_C=math.radians(111.2), CA_C_N=math.radians(116.2), C_N_CA=math.radians(121.7))
def place(a, b, c, length, angle, torsion):
    """NeRF: position of d with |cd| = length, ∠bcd = angle, dihedral(a, b, c, d) = torsion (batched)."""
    bc = c - b; bc /= np.linalg.norm(bc, axis=-1, keepdims=True)
    n = np.cross(b - a, bc); n /= np.linalg.norm(n, axis=-1, keepdims=True)
    m = np.stack([bc, np.cross(n, bc), n], -1)
    d2 = np.stack([-length * np.cos(angle) * np.ones_like(torsion), length * np.sin(angle) * np.cos(torsion),
                   length * np.sin(angle) * np.sin(torsion)], -1)
    return c + np.einsum("bij,bj->bi", m, d2)

def ideal_frames(tor):
    """tor: (B, n, 3) -> ideal-geometry frames (B, n+1, 3, 3) of the rebuilt residues."""
    B, n, _ = tor.shape
    N = np.zeros((B, 3)); CA = np.tile([BOND["N_CA"], 0, 0], (B, 1)).astype(float)
    C = CA + BOND["CA_C"] * np.stack([-np.cos(ANGLE["N_CA_C"]) * np.ones(B), np.sin(ANGLE["N_CA_C"]) * np.ones(B),
                                      np.zeros(B)], -1)
    atoms = [(N, CA, C)]
    for k in range(n):
        Nn = place(N, CA, C, BOND["C_N"], ANGLE["CA_C_N"], tor[:, k, 0])
        CAn = place(CA, C, Nn, BOND["N_CA"], ANGLE["C_N_CA"], tor[:, k, 1])
        Cn = place(C, Nn, CAn, BOND["CA_C"], ANGLE["N_CA_C"], tor[:, k, 2])
        N, CA, C = Nn, CAn, Cn; atoms.append((N, CA, C))
    X = np.stack([np.stack(a, 1) for a in atoms], 1)                     # (B, n+1, 3 atoms, 3)
    return frames(X.reshape(-1, 3, 3)).reshape(B, n + 1, 3, 3)

def ideal_prediction(starts, wrong_order=False):
    n = starts[0][2]; tor = np.stack([s["tor"][i:i + n] for s, i, _ in starts]).astype(float)
    R = ideal_frames(tor)
    if not wrong_order:
        Rel = np.einsum("bji,bjk->bik", R[:, 0], R[:, -1])
    else:
        steps = np.einsum("bnji,bnjk->bnik", R[:, :-1], R[:, 1:])          # per-residue rotations
        Rel = np.tile(np.eye(3), (len(starts), 1, 1))
        for k in range(n - 1, -1, -1): Rel = Rel @ steps[:, k]             # reverse order
    return torch.from_numpy(rot_to_quat(Rel).astype(np.float32))

## Training and evaluation

In [ ]:
def angle_deg(qp, qt):
    return torch.rad2deg(2 * torch.arccos(torch.clamp((qp * qt).sum(-1).abs(), max=1.0)))

@torch.no_grad()
def errors(model, win, bs=256):
    types, ang, L, Q, _ = win; out = []
    for i in range(0, len(L), bs):
        out.append(angle_deg(model(types[i:i+bs].to(DEVICE), ang[i:i+bs].to(DEVICE), L[i:i+bs].to(DEVICE)),
                             Q[i:i+bs].to(DEVICE)).cpu())
    return torch.cat(out).numpy()

val_set = make_windows("val", VAL_RES, N_VAL, np.random.default_rng(1))
longest = max(len(s_["X"]) for n_ in SPLITS["test"] for s_ in entries[n_])
TEST_RES = [n for n in TEST_RES if n < longest]                     # lengths the test proteins can supply
test_sets = {n: make_windows("test", n, N_TEST, np.random.default_rng(100 + n)) for n in TEST_RES}

def loss_fn(model, batch):
    types, ang, L, Q, _ = batch
    q = model(types.to(DEVICE), ang.to(DEVICE), L.to(DEVICE))
    return (1 - (q * Q.to(DEVICE)).sum(-1) ** 2).mean()

def multi_start(name, seed):
    """Products of rotations have local minima: try N_INIT initializations briefly on 1–2 residues and keep the one
    with the lowest training loss (same procedure for every model; validation and test data are not used)."""
    rng = np.random.default_rng(5000 + seed)
    probe = make_windows("train", (1, 2), 1024, rng)
    best, best_model = float("inf"), None
    for r in range(N_INIT):
        torch.manual_seed(1000 * seed + r)
        model = MAKE[name]().to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=LR[name])
        for _ in range(WARMUP_STEPS):
            loss = loss_fn(model, make_windows("train", (1, 2), BATCH, rng))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        with torch.no_grad(): l = loss_fn(model, probe).item()
        if l < best: best, best_model = l, model
    print(f"  {name:10s} seed {seed}: multi-start kept training loss {best:.5f} (of {N_INIT} starts)", flush=True)
    return best_model

def train(name, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(1000 + seed)
    model = multi_start(name, seed) if N_INIT > 1 else MAKE[name]().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR[name], weight_decay=0.0)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)
    best, best_state, t0 = float("inf"), None, time.time()
    for step in range(1, STEPS + 1):
        hi = 2 if step < CURRICULUM * STEPS else TRAIN_RES[1]
        loss = loss_fn(model, make_windows("train", (TRAIN_RES[0], hi), BATCH, rng))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if step % EVAL_EVERY == 0:
            model.eval(); val = float(np.median(errors(model, val_set))); model.train()
            if val < best: best, best_state = val, {k: v.detach().clone() for k, v in model.state_dict().items()}
            if step % (4 * EVAL_EVERY) == 0:
                print(f"  {name:10s} seed {seed} step {step:5d} loss {loss.item():.5f} val median "
                      f"{val:6.2f}° (best {best:6.2f}°) [{time.time()-t0:.0f}s]", flush=True)
    model.load_state_dict(best_state); model.eval()
    return model

In [ ]:
results = {}
results["no rotation"] = {n: angle_deg(torch.tensor([[1.0, 0, 0, 0]]), test_sets[n][3]).numpy() for n in TEST_RES}
results["ideal geometry"] = {n: angle_deg(ideal_prediction(test_sets[n][4]), test_sets[n][3]).numpy() for n in TEST_RES}
results["ideal, wrong order"] = {n: angle_deg(ideal_prediction(test_sets[n][4], True), test_sets[n][3]).numpy()
                                 for n in TEST_RES}
trained = {}
for name in MODELS:
    for seed in SEEDS:
        print(f"{name}, seed {seed}")
        m = train(name, seed); trained[(name, seed)] = m
        results[f"{name}|{seed}"] = {n: errors(m, test_sets[n]) for n in TEST_RES}
        print("  test median error (°): " + ", ".join(f"{n}: {np.median(results[f'{name}|{seed}'][n]):.1f}"
                                                      for n in TEST_RES))

## Results: median orientation error on held-out proteins against stretch length

In [ ]:
def summary(prefix):
    keys = [k for k in results if k == prefix or k.startswith(prefix + "|")]
    med = np.array([[np.median(results[k][n]) for n in TEST_RES] for k in keys])
    return med.mean(0), med.std(0)

ROWS = [r for r in ["no rotation", "ideal geometry", "ideal, wrong order"] + MODELS
        if any(k == r or k.startswith(r + "|") for k in results)]
print(f"median orientation error (°), held-out proteins; mean ± s.d. over {len(SEEDS)} seeds; "
      f"trained on {TRAIN_RES[0]}–{TRAIN_RES[1]} residues")
print(f"{'model':20s}" + "".join(f"{n:>12d}" for n in TEST_RES))
for r in ROWS:
    mu, sd = summary(r)
    print(f"{r:20s}" + "".join(f"{m:7.1f}±{d:<4.1f}" if d > 0 else f"{m:12.1f}" for m, d in zip(mu, sd)))
json.dump({r: [list(map(float, x)) for x in summary(r)] for r in ROWS} | {"lengths": TEST_RES},
          open("protein_results.json", "w"))

# figure in the paper's style
from matplotlib import font_manager
for f in font_manager.findSystemFonts():
    if any(k in f.lower() for k in ("heros", "helvet", "liberationsans")):
        try: font_manager.fontManager.addfont(f)
        except Exception: pass
plt.rcParams.update({"font.family": ["TeX Gyre Heros", "Liberation Sans", "DejaVu Sans"], "font.size": 8.5,
                     "axes.linewidth": 0.8, "axes.edgecolor": "#555555", "pdf.fonttype": 42})
BLUE, FILL, DARK, GREY, LIGHT = "#2a5ca8", "#e8eef8", "#3a3a3a", "#8a8a8a", "#b8b8b8"
STY = {"no rotation": dict(color=LIGHT, ls=(0, (6, 2, 1.2, 2)), lw=1.1),
       "ideal, wrong order": dict(color=GREY, ls=(0, (1.2, 1.6)), lw=1.3),
       "ideal geometry": dict(color=DARK, ls=(0, (4, 2)), lw=1.1),
       "C-diagonal": dict(color=DARK, lw=1.0, marker="s", ms=3.2, mfc="white", mew=0.9),
       "LSTM": dict(color=DARK, lw=1.0, marker="^", ms=3.6, mfc="white", mew=0.9),
       "HyperSSM": dict(color=BLUE, lw=2.0, marker="o", ms=3.4)}
LAB = {"HyperSSM": "HyperSSM, ℍ (4 numbers)", "C-diagonal": "ℂ diagonal (128 numbers)", "LSTM": "LSTM (128 units)",
       "ideal geometry": "ideal geometry", "ideal, wrong order": "ideal geometry, wrong order",
       "no rotation": "predict no rotation"}
fig, ax = plt.subplots(figsize=(3.4, 3.15))
ax.axvspan(TRAIN_RES[0], TRAIN_RES[1], color=FILL, lw=0, zorder=0)
for x in TRAIN_RES: ax.axvline(x, color=BLUE, lw=0.8, ls=(0, (3, 2)), zorder=0.5)
for r in [r for r in ["no rotation", "ideal, wrong order", "ideal geometry", "C-diagonal", "LSTM", "HyperSSM"] if r in ROWS]:
    mu, sd = summary(r)
    if sd.any(): ax.fill_between(TEST_RES, np.maximum(mu - sd, 1e-2), mu + sd, color=STY[r]["color"], alpha=0.14, lw=0)
    ax.plot(TEST_RES, mu, label=LAB[r], zorder=3 if r == "HyperSSM" else 2, **STY[r])
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xticks(TEST_RES); ax.set_xticklabels([str(n) for n in TEST_RES])
ax.set_xlabel("residues along the chain"); ax.set_ylabel("median orientation error (°)")
ax.text(4, ax.get_ylim()[1] * 0.75, "training", color=BLUE, ha="center", fontsize=7.5, fontweight="bold")
for s in ("top", "right"): ax.spines[s].set_visible(False)
order = [k for k in ["HyperSSM", "C-diagonal", "LSTM", "ideal geometry", "ideal, wrong order", "no rotation"] if k in ROWS]
h, l = ax.get_legend_handles_labels(); idx = [l.index(LAB[k]) for k in order]
fig.legend([h[i] for i in idx], [l[i] for i in idx], loc="lower center", ncol=2, frameon=False, fontsize=6.8,
           handlelength=2.4, columnspacing=1.2, labelspacing=0.35, bbox_to_anchor=(0.54, 0.0))
plt.tight_layout(pad=0.3, rect=(0, 0.17, 1, 1))
plt.savefig("protein_backbone.pdf"); plt.savefig("protein_backbone.png", dpi=400); plt.show()

## What HyperSSM learned
A torsion token turns the frame by `exp(α w)`: a rotation by `2|w|·α`. If the model learned the geometry, each
torsion axis has `|w| ≈ 0.5` (a rotation by exactly the torsion angle), and the bond tokens are fixed rotations
whose angles reflect the backbone's bond angles.

In [ ]:
m = trained[("HyperSSM", SEEDS[0])]
W = m.W.detach().cpu()
for t, nm in [(1, "ψ"), (3, "ω"), (5, "φ")]:
    print(f"torsion {nm}: rotation per radian of torsion = {2 * W[t, 1].norm():.4f} (ideal 1)")
for t, nm in [(0, "first bond"), (2, "bond after ψ"), (4, "bond after ω"), (6, "bond between residues"),
              (7, "last bond")]:
    print(f"{nm:22s}: rotation by {math.degrees(2 * W[t, 0].norm()):6.1f}°")